# 2. Silver Layer Transformation & Data Quality — Line-by-Line Breakdown

This notebook covers the cleaning, deduplication, schema normalization, and enrichment steps performed on the Bronze raw data to build clean, validated Silver tables in Delta Lake.

--- 
## Cell 1: Data Profiling — Inspecting Status Casing Anomalies

In [ ]:
%sql
-- Problem 1: the source system logs statuses in two casings
SELECT status, COUNT(*) AS row_count
FROM shopstream.core.bronze_orders
GROUP BY status
ORDER BY row_count DESC

### Line-by-Line Code Explanation:

1. **`%sql`**
   - Instructs Databricks to interpret cell contents as Spark SQL.

2. **`SELECT status, COUNT(*) AS row_count`**
   - **`status`**: Selects the order status text column from the raw table.
   - **`COUNT(*) AS row_count`**: Aggregates the total number of records associated with each unique status string, naming the aggregate alias `row_count`.

3. **`FROM shopstream.core.bronze_orders`**
   - Specifies the fully qualified Bronze target table containing the raw ingestion records.

4. **`GROUP BY status`**
   - Groups the dataset rows by distinct string values found inside the `status` column to reveal casing inconsistencies (e.g., `completed` vs `COMPLETED`).

5. **`ORDER BY row_count DESC`**
   - Sorts the grouped audit results in descending order by occurrence frequency.

--- 
## Cell 2: Data Profiling — Detecting Duplicate Line IDs & Invalid Quantities

In [ ]:
%sql
-- Problem 2: double-fired order lines and impossible quantities
SELECT
  (SELECT COUNT(*) FROM (
    SELECT order_line_id FROM shopstream.core.bronze_orders
    GROUP BY order_line_id HAVING COUNT(*) > 1
  )) AS duplicated_line_ids,
  (SELECT COUNT(*) FROM shopstream.core.bronze_orders WHERE quantity <= 0) AS bad_quantity_rows

### Line-by-Line Code Explanation:

1. **`SELECT`**
   - Begins the outer projection query that evaluates two distinct metric subqueries.

2. **`(SELECT COUNT(*) FROM (`**
   - Starts a scalar subquery to calculate the total count of duplicated `order_line_id` values.

3. **`SELECT order_line_id FROM shopstream.core.bronze_orders`**
   - Pulls `order_line_id` keys from the raw Bronze orders table.

4. **`GROUP BY order_line_id HAVING COUNT(*) > 1`**
   - Groups by line ID and applies a `HAVING` threshold to isolate line IDs that appear more than once in the table.

5. **`)) AS duplicated_line_ids,`**
   - Closes the inner inline view and names the resulting scalar metric `duplicated_line_ids`.

6. **`(SELECT COUNT(*) FROM shopstream.core.bronze_orders WHERE quantity <= 0) AS bad_quantity_rows`**
   - Evaluates a subquery counting records containing zero or negative quantities (`quantity <= 0`), aliasing the aggregate as `bad_quantity_rows`.

--- 
## Cell 3: Deduplicating, Cleaning, and Building `silver_orders`

In [ ]:
%sql
-- silver_orders: dedup double-fires, drop impossible rows, normalize, type, derive
CREATE OR REPLACE TABLE shopstream.core.silver_orders AS
WITH deduped AS (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY order_line_id ORDER BY order_ts) AS rn
  FROM shopstream.core.bronze_orders
)
SELECT
  order_line_id,
  order_id,
  customer_id,
  product_id,
  CAST(quantity AS INT) AS quantity,
  CAST(unit_price AS DOUBLE) AS unit_price,
  ROUND(quantity * unit_price, 2) AS line_revenue,
  CAST(order_ts AS TIMESTAMP) AS order_ts,
  LOWER(status) AS status,
  NULLIF(coupon_code, '') AS coupon_code
FROM deduped
WHERE rn = 1
  AND quantity > 0;

SELECT COUNT(*) AS silver_rows FROM shopstream.core.silver_orders

### Line-by-Line Code Explanation:

1. **`CREATE OR REPLACE TABLE shopstream.core.silver_orders AS`**
   - Atomically overwrites or instantiates the Delta table `silver_orders` inside Unity Catalog using the result set of the query below.

2. **`WITH deduped AS (`**
   - Defines a Common Table Expression (CTE) named `deduped` to handle deduplication pre-processing.

3. **`SELECT *, ROW_NUMBER() OVER (PARTITION BY order_line_id ORDER BY order_ts) AS rn`**
   - **`ROW_NUMBER()`**: Generates an incremental integer index for each record.
   - **`PARTITION BY order_line_id`**: Isolates rows into independent windows grouped by unique `order_line_id`.
   - **`ORDER BY order_ts`**: Ranks rows chronologically within each window partition.

4. **`FROM shopstream.core.bronze_orders`**
   - Reads raw input rows from the Bronze layer.

5. **`SELECT order_line_id, order_id, customer_id, product_id,`**
   - Selects primary relational identifiers.

6. **`CAST(quantity AS INT) AS quantity, CAST(unit_price AS DOUBLE) AS unit_price,`**
   - Explicitly casts string attributes to formal numeric types (`INT` and `DOUBLE`).

7. **`ROUND(quantity * unit_price, 2) AS line_revenue,`**
   - Calculates derived line item financial value rounded to two decimal places.

8. **`CAST(order_ts AS TIMESTAMP) AS order_ts,`**
   - Ensures uniform timestamp formatting.

9. **`LOWER(status) AS status,`**
   - Standardizes status string casing across all entries to lower-case values (`COMPLETED` → `completed`).

10. **`NULLIF(coupon_code, '') AS coupon_code`**
    - Converts empty strings (`''`) into explicit SQL `NULL` values for clean data processing.

11. **`FROM deduped WHERE rn = 1 AND quantity > 0;`**
    - Keeps only the first occurrence (`rn = 1`) per duplicated line ID and purges bad input entries (`quantity <= 0`).

12. **`SELECT COUNT(*) AS silver_rows FROM shopstream.core.silver_orders`**
    - Evaluates total valid rows written into the Silver orders table.

--- 
## Cell 4: Cleaning & Enriching Dimension Tables (`Customers` & `Products`)

In [ ]:
%sql
-- Dimensions need a lighter touch: types + a helpful derived column
CREATE OR REPLACE TABLE shopstream.core.silver_customers AS
SELECT
  customer_id,
  name,
  email,
  city,
  country,
  CAST(signup_date AS DATE) AS signup_date,
  signup_channel
FROM shopstream.core.bronze_customers;

CREATE OR REPLACE TABLE shopstream.core.silver_products AS
SELECT
  product_id,
  product_name,
  category,
  CAST(unit_price AS DOUBLE) AS unit_price,
  CAST(unit_cost AS DOUBLE) AS unit_cost,
  ROUND(unit_price - unit_cost, 2) AS unit_margin
FROM shopstream.core.bronze_products;

### Line-by-Line Code Explanation:

1. **`CREATE OR REPLACE TABLE shopstream.core.silver_customers AS`**
   - Rebuilds the cleaned customer dimension table.

2. **`CAST(signup_date AS DATE) AS signup_date`**
   - Converts customer registration date strings into strong `DATE` data types.

3. **`CREATE OR REPLACE TABLE shopstream.core.silver_products AS`**
   - Rebuilds the cleaned product catalog table.

4. **`ROUND(unit_price - unit_cost, 2) AS unit_margin`**
   - Computes profit margin per unit directly in the product dimension table for fast downstream aggregation.

--- 
## Cell 5: Auditing Delta Lake Transaction History (`DESCRIBE HISTORY`)

In [ ]:
%sql
-- Delta keeps a transaction log for every table. Look:
DESCRIBE HISTORY shopstream.core.silver_orders

### Line-by-Line Code Explanation:

1. **`DESCRIBE HISTORY shopstream.core.silver_orders`**
   - Queries the Delta Lake transaction log (`_delta_log`) for `silver_orders`.
   - Returns audit metadata including version history, timestamps, user identities, commit operations, and underlying written Parquet file counts.

--- 
## Cell 6: Applying Business Deletions & Delta Time Travel Audit

In [ ]:
%sql
-- Business rule: cancelled orders don't belong in silver.
-- Delta DML makes this a one-liner, and time travel keeps the audit trail.
DELETE FROM shopstream.core.silver_orders WHERE status = 'cancelled';

SELECT
  (SELECT COUNT(*) FROM shopstream.core.silver_orders) AS rows_now,
  (SELECT COUNT(*) FROM shopstream.core.silver_orders VERSION AS OF 0) AS rows_before_delete

### Line-by-Line Code Explanation:

1. **`DELETE FROM shopstream.core.silver_orders WHERE status = 'cancelled';`**
   - **`DELETE FROM`**: Executes a Delta Lake DML mutation statement that removes matching records from the active state of the table.
   - **`WHERE status = 'cancelled'`**: Applies the business rule specifying that cancelled transactions should not reside in the active Silver layer.

2. **`(SELECT COUNT(*) FROM shopstream.core.silver_orders) AS rows_now`**
   - Counts total records remaining in the active, current version of `silver_orders`.

3. **`(SELECT COUNT(*) FROM shopstream.core.silver_orders VERSION AS OF 0) AS rows_before_delete`**
   - **`VERSION AS OF 0`**: Leverages Delta Lake **Time Travel** capabilities to query historical snapshot state (`Version 0`) prior to the delete transaction, verifying audit traceability.